# MorphoGAT: external prediction and spatial readouts

Run Notebook 01 first, or point the variables below to an existing checkpoint and manifest. Prediction requires RNA, coordinates and valid images according to the saved configuration; it never reads a protein target file.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "morphogat").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "morphogat").is_dir(), "Open this notebook inside the source checkout"

def run(*arguments):
    subprocess.run([sys.executable, "-m", "morphogat", *map(str, arguments)], cwd=ROOT, check=True)


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Image, display

DATA = ROOT / "data" / "notebook_demo"
MODEL = ROOT / "runs" / "notebook_model"
OUT = ROOT / "runs" / "notebook_external"
run("predict", "--checkpoint", MODEL / "best.pt", "--manifest", DATA / "manifest.yaml",
    "--split", "external", "--output", OUT, "--device", "cpu")
PREDICTIONS = OUT / "demo_4" / "predictions.npz"
pd.read_csv(OUT / "demo_4" / "predicted_protein.csv").head()


## Protein, graph weight and fusion gate maps

These readouts describe relative input use. The toy image and counts do not establish a biological interpretation.

In [ ]:
run("plot-spatial", "--predictions", PREDICTIONS, "--protein", "CD3", "--output", OUT / "CD3.png")
run("plot-spatial", "--predictions", PREDICTIONS, "--readout", "morphology", "--output", OUT / "morphology.png")
run("plot-spatial", "--predictions", PREDICTIONS, "--readout", "mean_gate", "--output", OUT / "gate.png")
for name in ["CD3.png", "morphology.png", "gate.png"]:
    display(Image(filename=str(OUT / name), width=550))


In [ ]:
weights = pd.read_csv(OUT / "demo_4" / "view_weights_and_gates.csv", index_col=0)
assert np.allclose(weights[["spatial", "rna", "morphology"]].sum(axis=1), 1)
weights.describe()


## Optional spatial domain analysis

Install `requirements-analysis.txt`. On study data use `cluster --predictions ... --labels ... --resolution ... --neighbors ... --output ...`. Annotation CSVs contain `barcode,label`. Set clustering parameters without optimizing against held-out labels; the original figure settings must be supplied for exact reproduction.